# Neural Networks Refresh: MLP Regressor — Energy Efficiency
## Class Example — Medina County Career Center

**Goal:** Predict a building's heating load from its design specs using a neural network.

**Dataset:** Energy Efficiency from UCI (768 buildings, 8 features, no missing values)

**The process (same as regression + one new step):**
1. Load the data
2. Explore
3. Train/test split
4. **SCALE the data** ← NEW! Neural networks need this
5. Build the MLP Regressor
6. Evaluate (R², MAE)
7. Compare to Linear Regression

## Step 0: Install + Import

In [ ]:
!pip install ucimlrepo -q

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler          # NEW: scales data for NN
from sklearn.neural_network import MLPRegressor            # the neural network
from sklearn.linear_model import LinearRegression          # for comparison
from sklearn.metrics import r2_score, mean_absolute_error
from ucimlrepo import fetch_ucirepo
import warnings
warnings.filterwarnings('ignore')  # suppress convergence warnings

print('All libraries loaded!')

## Step 1: Load the Data

This dataset simulates 768 buildings with different designs. We predict the **Heating Load** — how much energy the building needs to heat.

Features include things like surface area, wall area, roof area, glazing area, etc.

In [ ]:
# Fetch Energy Efficiency dataset from UCI (id=242)
energy = fetch_ucirepo(id=242)

X = energy.data.features
yFull = energy.data.targets

# This dataset has 2 targets: Heating Load and Cooling Load
# We'll predict just Heating Load (Y1)
y = yFull.iloc[:, 0]  # first target column = Heating Load

print(f'Dataset: {len(X)} buildings, {X.shape[1]} features')
print(f'Target: Heating Load')
print(f'\nFeatures:')
for col in X.columns:
    print(f'  - {col}')
print(f'\nFirst 5 rows:')
X.head()

In [ ]:
# Check for missing values (spoiler: there are none!)
print('Missing values:', X.isnull().sum().sum())
print('\nBasic statistics:')
X.describe().round(2)

## Step 2: Explore — Correlations with Heating Load

In [ ]:
# Combine for correlation analysis
df = pd.concat([X, y.rename('Heating_Load')], axis=1)

# Correlations with target
corrs = df.corr()['Heating_Load'].drop('Heating_Load').sort_values()
print('Correlations with Heating Load:\n')
for feature, r in corrs.items():
    print(f'  {feature:25s}  r = {r:+.3f}  R² = {r**2:.3f}')

In [ ]:
# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap — Energy Efficiency', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 3: Train/Test Split

In [ ]:
# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training: {len(X_train)} buildings')
print(f'Test:     {len(X_test)} buildings')

## Step 4: SCALE the Data (New Step!)

Neural networks need all features on the same scale. `StandardScaler` transforms each feature so the mean = 0 and standard deviation = 1.

**Important:** Fit the scaler on TRAINING data only, then apply the same transformation to test data.

In [ ]:
# Create the scaler
scaler = StandardScaler()

# Fit on training data, then transform both
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform
X_test_scaled = scaler.transform(X_test)          # transform only (use same scaling)

print('Before scaling (first row):')
print(f'  {X_train.iloc[0].values}')
print(f'\nAfter scaling (first row):')
print(f'  {X_train_scaled[0].round(3)}')
print(f'\nAll values now centered around 0 — much better for neural networks!')

## Step 5: Build the MLP Regressor

MLP = Multi-Layer Perceptron. It's a neural network!

- `hidden_layer_sizes=(64, 32)` → 2 hidden layers with 64 and 32 neurons
- `max_iter=500` → train for up to 500 rounds
- Uses SCALED data

In [ ]:
# Build and train the neural network
mlpModel = MLPRegressor(
    hidden_layer_sizes=(64, 32),   # 2 hidden layers: 64 neurons, then 32
    max_iter=500,                  # train up to 500 iterations
    random_state=42
)

print('Training the neural network...')
mlpModel.fit(X_train_scaled, y_train)
print(f'Done! Trained for {mlpModel.n_iter_} iterations.')
print(f'\nArchitecture: Input({X_train.shape[1]}) → Hidden(64) → Hidden(32) → Output(1)')

## Step 6: Evaluate the Model

In [ ]:
# Predictions on test set (use SCALED test data!)
mlpPredictions = mlpModel.predict(X_test_scaled)

mlpR2 = r2_score(y_test, mlpPredictions)
mlpMAE = mean_absolute_error(y_test, mlpPredictions)

print('MLP REGRESSOR PERFORMANCE:')
print(f'  R² Score: {mlpR2:.4f} ({mlpR2*100:.1f}%)')
print(f'  MAE:      {mlpMAE:.2f} kWh/m²')
print(f'\nThe model explains {mlpR2*100:.1f}% of heating load variation.')
print(f'Predictions are off by about {mlpMAE:.1f} kWh/m² on average.')

## Step 7: Compare to Linear Regression

Is the neural network actually better than plain regression for this problem?

In [ ]:
# Build Linear Regression for comparison (does NOT need scaling)
lrModel = LinearRegression()
lrModel.fit(X_train, y_train)
lrPredictions = lrModel.predict(X_test)

lrR2 = r2_score(y_test, lrPredictions)
lrMAE = mean_absolute_error(y_test, lrPredictions)

# Side by side comparison
print('=' * 55)
print('COMPARISON: Linear Regression vs MLP Neural Network')
print('=' * 55)
print(f'{"Model":<30} {"R²":<12} {"MAE":<10}')
print('-' * 55)
print(f'{"Linear Regression":<30} {lrR2:<12.4f} {lrMAE:<10.2f}')
print(f'{"MLP Regressor (64-32)":<30} {mlpR2:<12.4f} {mlpMAE:<10.2f}')
print('=' * 55)

if mlpR2 > lrR2:
    print(f'\nNeural network wins! R² improved by {(mlpR2 - lrR2)*100:.1f} percentage points.')
else:
    print(f'\nLinear regression wins for this dataset.')
    print('This can happen when relationships are mostly linear.')

In [ ]:
# Visual comparison: Actual vs Predicted for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear Regression plot
axes[0].scatter(y_test, lrPredictions, alpha=0.5, color='steelblue', s=30)
minVal = min(y_test.min(), lrPredictions.min())
maxVal = max(y_test.max(), lrPredictions.max())
axes[0].plot([minVal, maxVal], [minVal, maxVal], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Heating Load')
axes[0].set_ylabel('Predicted Heating Load')
axes[0].set_title(f'Linear Regression (R²={lrR2:.3f})', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# MLP plot
axes[1].scatter(y_test, mlpPredictions, alpha=0.5, color='darkorange', s=30)
minVal = min(y_test.min(), mlpPredictions.min())
maxVal = max(y_test.max(), mlpPredictions.max())
axes[1].plot([minVal, maxVal], [minVal, maxVal], 'r--', linewidth=2)
axes[1].set_xlabel('Actual Heating Load')
axes[1].set_ylabel('Predicted Heating Load')
axes[1].set_title(f'MLP Neural Network (R²={mlpR2:.3f})', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Linear Regression vs Neural Network', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Closer to the red line = better predictions')

## Step 8: Experiment — Try Different Architectures

Change the hidden layer sizes and see what happens!

In [ ]:
# Try several architectures and compare
architectures = [
    (10,),           # small: 1 layer, 10 neurons
    (64, 32),        # medium: 2 layers
    (128, 64, 32),   # large: 3 layers
]

print(f'{"Architecture":<25} {"R²":<12} {"MAE":<10} {"Iterations":<10}')
print('-' * 60)

for arch in architectures:
    tempModel = MLPRegressor(hidden_layer_sizes=arch, max_iter=500, random_state=42)
    tempModel.fit(X_train_scaled, y_train)
    tempPred = tempModel.predict(X_test_scaled)
    tempR2 = r2_score(y_test, tempPred)
    tempMAE = mean_absolute_error(y_test, tempPred)
    print(f'{str(arch):<25} {tempR2:<12.4f} {tempMAE:<10.2f} {tempModel.n_iter_:<10}')

# Also show Linear Regression for reference
print(f'{"Linear Regression":<25} {lrR2:<12.4f} {lrMAE:<10.2f} {"N/A":<10}')

print('\nBigger is not always better! The best architecture depends on the data.')

---

## Summary

**What we did:**
1. Loaded building energy data from UCI (768 buildings, 8 features)
2. Explored correlations
3. Split 80/20 for honest evaluation
4. **SCALED the data** with StandardScaler (critical for neural networks!)
5. Built an MLP Regressor (neural network) with 2 hidden layers
6. Compared to Linear Regression side-by-side
7. Experimented with different architectures

**Key differences from plain regression:**
- Must SCALE the data (StandardScaler)
- Use `MLPRegressor` instead of `LinearRegression`
- Specify architecture with `hidden_layer_sizes`
- Everything else is the SAME process

**Remember:**
- `MLPRegressor` → predicts a NUMBER
- `MLPClassifier` → predicts a CATEGORY
- Always compare to a simpler model first!